# 06 – Integration benchmark on Colab T4 (M8)

Runs the packaged FetalVision pipeline on a T4: re-export + parity, benchmark with `CUDAExecutionProvider` and CPU (tag `colab_t4`), start the HTTP service and call it, show an annotated output.

Runtime: **GPU (T4)**. No logic lives here: every step calls a CLI from the package (see `README.md`).

In [ ]:
import os, subprocess, glob
# Code source: (a) an uploaded bundle fetalvision_colab*.zip in /content (upload it with the file pane or the cell below),
# or (b) the team repository once the v2 code is pushed there.
parts = sorted(glob.glob('/content/fetalvision_colab.zip.part*'))
if parts and not os.path.exists('/content/fetalvision_colab.zip'):
    !cat /content/fetalvision_colab.zip.part* > /content/fetalvision_colab.zip
bundle = sorted(glob.glob('/content/fetalvision_colab.zip'))
if not os.path.exists('/content/FetalVision/pyproject.toml'):
    if bundle:
        !rm -rf /content/FetalVision && mkdir -p /content/FetalVision && unzip -q -o {bundle[0]} -d /content/FetalVision
    else:
        from google.colab import files
        print('Upload fetalvision_colab.zip (code + ONNX models) or press cancel to clone the team repository')
        up = files.upload()
        if up:
            name = list(up)[0]; os.rename(name, '/content/fetalvision_colab.zip')
            !rm -rf /content/FetalVision && mkdir -p /content/FetalVision && unzip -q -o /content/fetalvision_colab.zip -d /content/FetalVision
        else:
            !git clone -q --depth 1 https://github.com/lamooooo1418/FetalVision.git /content/FetalVision
%cd /content/FetalVision
!ls
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# dependencies: Colab already ships TensorFlow + PyTorch (CUDA). Add only what the pipeline/service/benchmark need.
!pip install -q onnx onnxruntime-gpu fastapi uvicorn httpx python-multipart imagehash pyyaml psutil opencv-python-headless jinja2
!pip install -q -e . --no-deps
import onnxruntime as ort; print('ORT', ort.__version__, 'providers:', ort.get_available_providers())

In [ ]:
# data: 32 HC18 images for parity/benchmark (HC18 training set, Zenodo 1327317, CC BY 4.0, 133 MB)
import os
os.environ['FETALVISION_DATA'] = '/content/data'
!mkdir -p $FETALVISION_DATA/hc18/raw
!test -f $FETALVISION_DATA/hc18/raw/training_set.zip || wget -q -O $FETALVISION_DATA/hc18/raw/training_set.zip "https://zenodo.org/records/1327317/files/training_set.zip?download=1"
!cd $FETALVISION_DATA/hc18/raw && mkdir -p parity_samples && unzip -l training_set.zip | awk '{print $4}' | grep -E '^training_set/[0-9]+_HC\.png$' | sort | awk 'NR%25==1' | head -32 > /tmp/list.txt && unzip -o -j -q training_set.zip $(cat /tmp/list.txt) -d parity_samples && ls parity_samples | wc -l

In [ ]:
# models: the exported ONNX files are committed in models/. The native sources (Keras / .pt) are needed only for parity
# and the native backend; put them in models/ (e.g. from Drive) or skip parity below.
!ls -la models/ | head -20

In [ ]:
# re-export + parity only when the native training weights are present (they are git-ignored); otherwise the committed ONNX models are used as-is.
import os
if all(os.path.exists(p) for p in ['models/head_detector.pt', 'models/head_segmenter.pt']):
    !bash scripts/export_all.sh configs/export.yaml --providers CUDAExecutionProvider,CPUExecutionProvider
else:
    print('native weights not present -> skipping re-export; benchmarking the committed ONNX models in models/')

In [ ]:
# benchmark: ORT CUDA (T4) + ORT CPU (Colab 2 vCPU); tag colab_t4
!python -m fetalvision.bench.benchmark --config configs/service.yaml --out results/phase2/benchmark/colab_t4 \
    --backends ort_cuda,ort_cpu --n-warmup 20 --n-iters 200 --http --http-requests 100
from IPython.display import Markdown, Image, display
display(Markdown(open('results/phase2/benchmark/colab_t4/results_table.md').read()))
display(Image('results/phase2/benchmark/colab_t4/figures/latency_by_stage.png'))

In [ ]:
# acceptance criteria (T4: p95 <= 200 ms, >= 5 img/s; CPU: p95 <= 1500 ms)
import json; b = json.load(open('results/phase2/benchmark/colab_t4/benchmark.json'))
print(json.dumps(b['acceptance'], indent=2)); print('stand-in models:', b['stand_in_models'])

In [ ]:
# start the service in the background (CUDA provider via a copy of the config) and call it
import yaml, subprocess, time, httpx
cfg = yaml.safe_load(open('configs/service.yaml')); cfg['runtime']['providers'] = ['CUDAExecutionProvider', 'CPUExecutionProvider']
yaml.safe_dump(cfg, open('/tmp/service_cuda.yaml', 'w'), sort_keys=False)
env = dict(os.environ, FETALVISION_SERVICE_CONFIG='/tmp/service_cuda.yaml')
proc = subprocess.Popen(['uvicorn', 'fetalvision.service.app:app', '--host', '127.0.0.1', '--port', '8000', '--workers', '1'], env=env)
for _ in range(120):
    try:
        if httpx.get('http://127.0.0.1:8000/health', timeout=2).status_code == 200: break
    except Exception: time.sleep(1)
print(httpx.get('http://127.0.0.1:8000/health').json())

In [ ]:
import glob
img = sorted(glob.glob(os.environ['FETALVISION_DATA'] + '/hc18/raw/parity_samples/*.png'))[0]
r = httpx.post('http://127.0.0.1:8000/predict', files={'file': open(img, 'rb')}, data={'pixel_size_mm': '0.0691', 'force_head': 'true'})
print(json.dumps(r.json(), indent=1)[:1500])
r = httpx.post('http://127.0.0.1:8000/predict/annotated', files={'file': open(img, 'rb')}, data={'pixel_size_mm': '0.0691', 'force_head': 'true'})
open('/tmp/annotated.png', 'wb').write(r.content); print(dict(r.headers)); display(Image('/tmp/annotated.png'))
print(httpx.get('http://127.0.0.1:8000/metrics').json())

In [ ]:
proc.terminate(); proc.wait(timeout=10)
# results to commit: results/phase2/benchmark/colab_t4/{metrics.json,benchmark.json,results_table.md,figures/}, results/phase2/export/parity_*.json
!ls -R results/phase2/benchmark/colab_t4 results/phase2/export | head -30

In [ ]:
# download the benchmark results to commit them into the repository (results/phase2/benchmark/colab_t4/)
!zip -q -r /content/colab_t4_benchmark.zip results/phase2/benchmark/colab_t4
from google.colab import files; files.download('/content/colab_t4_benchmark.zip')